# Phase 3: Comprehensive Image Transformation & Deep Learning Analysis

## Complete Benchmark Study Results

**Author:** EEG2Img-Benchmark Study  
**Date:** 2026-04-12  
**Phase:** 3 - Image Transformation & Deep Learning  

---

## Table of Contents

1. [Introduction & Objectives](#1-introduction)
2. [Data Loading & Preprocessing](#2-data-loading)
3. [Transform Method Analysis](#3-transform-analysis)
4. [Model Architecture Comparison](#4-model-comparison)
5. [Statistical Analysis](#5-statistical-analysis)
6. [Robustness Evaluation](#6-robustness)
7. [Best Practices & Recommendations](#7-recommendations)
8. [Conclusions](#8-conclusions)
9. [Publication-Ready Figures](#9-figures)
10. [Appendices](#10-appendices)

## 1. Introduction & Objectives {#1-introduction}

### Research Questions

1. **Which time-series-to-image transformation is most effective for EEG motor imagery classification?**
2. **How do CNN and Vision Transformer architectures compare for EEG image classification?**
3. **Do image-based methods outperform raw time-series baselines?**
4. **How robust are image-based methods to noise, channel dropout, and temporal jitter?**

### Experimental Design

- **Transforms:** 6 methods (GAF, MTF, RP, STFT, CWT, Topographic)
- **Models:** 7 architectures (ResNet-18/34/50, ViT-Base/Small, Lightweight CNN, + 3 baselines)
- **Validation:** 5-fold stratified cross-validation
- **Dataset:** BCI Competition IV-2a (4-class motor imagery)
- **Subjects:** 9 healthy participants
- **Metrics:** Accuracy, precision, recall, F1-score, confusion matrix

In [ ]:
# Setup and imports
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon, friedmanchisquare, ttest_rel
import json
import warnings
warnings.filterwarnings('ignore')

# Set paths
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'results' / 'phase3'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Plotting style
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10

print("✓ Environment setup complete")
print(f"Project root: {PROJECT_ROOT}")
print(f"Results directory: {RESULTS_DIR}")

## 2. Data Loading & Preprocessing {#2-data-loading}

Load and consolidate results from all experiments.

In [ ]:
# Load aggregated results
results_csv = RESULTS_DIR / 'aggregated_results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)
    print(f"✓ Loaded {len(df)} experiments")
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
else:
    print("⚠ Results file not found. Run 06_aggregate_results.py first.")
    df = None

# Display first few rows
if df is not None:
    display(df.head())

In [ ]:
# Summary statistics
if df is not None:
    print("="*60)
    print("EXPERIMENT SUMMARY")
    print("="*60)
    print(f"Total experiments: {len(df)}")
    print(f"\nUnique transforms: {df['transform'].nunique()}")
    print(f"Transforms: {sorted(df['transform'].unique())}")
    print(f"\nUnique models: {df['model'].nunique()}")
    print(f"Models: {sorted(df['model'].unique())}")
    print(f"\nUnique subjects: {df['subject'].nunique()}")
    print(f"Subjects: {sorted(df['subject'].unique())}")
    print(f"\nAccuracy range: {df['mean_accuracy'].min():.2f}% - {df['mean_accuracy'].max():.2f}%")
    print(f"Mean accuracy: {df['mean_accuracy'].mean():.2f} ± {df['mean_accuracy'].std():.2f}%")

## 3. Transform Method Analysis {#3-transform-analysis}

### 3.1 Overall Transform Performance

In [ ]:
# Group by transform
if df is not None:
    transform_stats = df.groupby('transform').agg({
        'mean_accuracy': ['mean', 'std', 'min', 'max', 'count']
    }).round(2)
    
    transform_stats.columns = ['_'.join(col).strip('_') for col in transform_stats.columns]
    transform_stats = transform_stats.sort_values('mean_accuracy_mean', ascending=False)
    
    print("\nTransform Performance Summary:")
    print("="*60)
    display(transform_stats)

In [ ]:
# Visualize transform comparison
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar plot
    transform_means = df.groupby('transform')['mean_accuracy'].mean().sort_values(ascending=False)
    transform_means.plot(kind='bar', ax=axes[0], color=sns.color_palette('husl', len(transform_means)))
    axes[0].set_ylabel('Mean Accuracy (%)')
    axes[0].set_title('Mean Accuracy by Transform Method')
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_xlabel('')
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Box plot
    transforms = sorted(df['transform'].unique())
    transform_data = [df[df['transform'] == t]['mean_accuracy'].values for t in transforms]
    bp = axes[1].boxplot(transform_data, labels=transforms, patch_artist=True)
    
    colors = sns.color_palette('husl', len(transforms))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Accuracy Distribution by Transform')
    axes[1].grid(True, alpha=0.3, axis='y')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'transform_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Figure saved: transform_comparison.png")

### 3.2 Statistical Significance Between Transforms

In [ ]:
# Pairwise statistical tests
if df is not None:
    transforms = sorted(df['transform'].unique())
    n_transforms = len(transforms)
    p_matrix = np.ones((n_transforms, n_transforms))
    
    for i, t1 in enumerate(transforms):
        for j, t2 in enumerate(transforms):
            if i < j:
                acc1 = df[df['transform'] == t1]['mean_accuracy'].values
                acc2 = df[df['transform'] == t2]['mean_accuracy'].values
                
                min_len = min(len(acc1), len(acc2))
                if min_len > 1:
                    try:
                        _, p = wilcoxon(acc1[:min_len], acc2[:min_len])
                        p_matrix[i, j] = p
                        p_matrix[j, i] = p
                    except:
                        pass
    
    # Plot heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(p_matrix, annot=True, fmt='.3f', cmap='RdYlGn_r',
                xticklabels=transforms, yticklabels=transforms,
                vmin=0, vmax=0.05, center=0.025,
                cbar_kws={'label': 'p-value'})
    plt.title('Statistical Significance Matrix (Wilcoxon Test)\nTransforms Comparison')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'transform_significance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Figure saved: transform_significance.png")

## 4. Model Architecture Comparison {#4-model-comparison}

### 4.1 Model Performance Analysis

In [ ]:
# Group by model
if df is not None:
    model_stats = df.groupby('model').agg({
        'mean_accuracy': ['mean', 'std', 'min', 'max', 'count']
    }).round(2)
    
    model_stats.columns = ['_'.join(col).strip('_') for col in model_stats.columns]
    model_stats = model_stats.sort_values('mean_accuracy_mean', ascending=False)
    
    print("\nModel Performance Summary:")
    print("="*60)
    display(model_stats)

In [ ]:
# Visualize model comparison
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar plot
    model_means = df.groupby('model')['mean_accuracy'].mean().sort_values(ascending=False)
    model_means.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', len(model_means)))
    axes[0].set_ylabel('Mean Accuracy (%)')
    axes[0].set_title('Mean Accuracy by Model Architecture')
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_xlabel('')
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Box plot
    models = sorted(df['model'].unique())
    model_data = [df[df['model'] == m]['mean_accuracy'].values for m in models]
    bp = axes[1].boxplot(model_data, labels=models, patch_artist=True)
    
    colors = sns.color_palette('Set2', len(models))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Accuracy Distribution by Model')
    axes[1].grid(True, alpha=0.3, axis='y')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Figure saved: model_comparison.png")

### 4.2 Transform × Model Interaction Heatmap

In [ ]:
# Create interaction heatmap
if df is not None:
    pivot = df.pivot_table(values='mean_accuracy', index='transform', columns='model', aggfunc='mean')
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
                center=75, vmin=60, vmax=90,
                cbar_kws={'label': 'Mean Accuracy (%)'})
    plt.title('Mean Classification Accuracy: Transform × Model Interaction')
    plt.xlabel('Model Architecture')
    plt.ylabel('Transform Method')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'interaction_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Figure saved: interaction_heatmap.png")

## 5. Statistical Analysis {#5-statistical-analysis}

### 5.1 Top Performers

In [ ]:
# Top 10 performers
if df is not None:
    top_10 = df.nlargest(10, 'mean_accuracy')[[
        'transform', 'model', 'subject', 'mean_accuracy', 'std_accuracy'
    ]].copy()
    
    top_10['rank'] = range(1, len(top_10) + 1)
    top_10 = top_10[['rank', 'transform', 'model', 'subject', 'mean_accuracy', 'std_accuracy']]
    
    print("\nTop 10 Performing Methods:")
    print("="*80)
    display(top_10)
    
    # Save to CSV
    top_10.to_csv(RESULTS_DIR / 'top_10_performers.csv', index=False)
    print(f"\n✓ Top 10 saved to: top_10_performers.csv")

### 5.2 ANOVA: Main Effects Analysis

In [ ]:
# Two-way ANOVA
if df is not None:
    from scipy.stats import f_oneway
    
    print("\nANOVA: Main Effects")
    print("="*60)
    
    # Effect of Transform
    transform_groups = [df[df['transform'] == t]['mean_accuracy'].values 
                       for t in df['transform'].unique()]
    f_stat_transform, p_transform = f_oneway(*transform_groups)
    print(f"Transform Effect: F={f_stat_transform:.2f}, p={p_transform:.4f}")
    
    # Effect of Model
    model_groups = [df[df['model'] == m]['mean_accuracy'].values 
                   for m in df['model'].unique()]
    f_stat_model, p_model = f_oneway(*model_groups)
    print(f"Model Effect: F={f_stat_model:.2f}, p={p_model:.4f}")
    
    # Interpret results
    print("\nInterpretation:")
    if p_transform < 0.05:
        print("✓ Transform method has a SIGNIFICANT effect on accuracy")
    else:
        print("✗ Transform method has NO significant effect on accuracy")
    
    if p_model < 0.05:
        print("✓ Model architecture has a SIGNIFICANT effect on accuracy")
    else:
        print("✗ Model architecture has NO significant effect on accuracy")

### 5.3 Effect Size Analysis (Cohen's d)

In [ ]:
def cohens_d(group1, group2):
    """Calculate Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std

if df is not None:
    # Compare best transform vs others
    best_transform = transform_means.index[0]
    best_acc = df[df['transform'] == best_transform]['mean_accuracy'].values
    
    print(f"\nEffect Size: {best_transform} vs Other Transforms")
    print("="*60)
    
    for transform in df['transform'].unique():
        if transform != best_transform:
            other_acc = df[df['transform'] == transform]['mean_accuracy'].values
            d = cohens_d(best_acc, other_acc)
            print(f"{best_transform} vs {transform:20s}: d = {d:+.3f} ", end='')
            
            if abs(d) < 0.2:
                print("(negligible)")
            elif abs(d) < 0.5:
                print("(small)")
            elif abs(d) < 0.8:
                print("(medium)")
            else:
                print("(large)")

## 6. Robustness Evaluation {#6-robustness}

Placeholder for robustness test results (noise, dropout, jitter).

In [ ]:
# Load robustness results if available
robustness_file = RESULTS_DIR / 'robustness' / 'robustness_results_all.json'

if robustness_file.exists():
    with open(robustness_file, 'r') as f:
        robustness_data = json.load(f)
    print("✓ Robustness results loaded")
    # Plot robustness curves here
else:
    print("⚠ Robustness results not found. Run 07_robustness_tests.py first.")

## 7. Best Practices & Recommendations {#7-recommendations}

Based on the analysis, here are the key findings and recommendations:

In [ ]:
if df is not None:
    print("="*80)
    print("KEY FINDINGS & RECOMMENDATIONS")
    print("="*80)
    
    # Best transform
    best_transform = transform_means.index[0]
    print(f"\n1. BEST TRANSFORM METHOD: {best_transform}")
    print(f"   Mean Accuracy: {transform_means.iloc[0]:.2f}%")
    
    # Best model
    best_model = model_means.index[0]
    print(f"\n2. BEST MODEL ARCHITECTURE: {best_model}")
    print(f"   Mean Accuracy: {model_means.iloc[0]:.2f}%")
    
    # Best combination
    best_combo = df.loc[df['mean_accuracy'].idxmax()]
    print(f"\n3. BEST OVERALL COMBINATION:")
    print(f"   Transform: {best_combo['transform']}")
    print(f"   Model: {best_combo['model']}")
    print(f"   Subject: {best_combo['subject']}")
    print(f"   Accuracy: {best_combo['mean_accuracy']:.2f} ± {best_combo['std_accuracy']:.2f}%")
    
    print("\n4. RECOMMENDATIONS:")
    print("   - Use {} for optimal performance".format(best_transform))
    print("   - {} architecture provides best accuracy".format(best_model))
    print("   - Consider computational cost vs accuracy trade-off")
    print("   - Validate robustness before deployment")

## 8. Conclusions {#8-conclusions}

### Summary of Findings

This comprehensive benchmark study evaluated 6 time-series-to-image transformation methods combined with 7 deep learning architectures for EEG motor imagery classification.

**Key Conclusions:**

1. **Transform Methods:** [Results to be filled based on actual data]
2. **Model Architectures:** [Results to be filled based on actual data]
3. **Statistical Significance:** [Results to be filled based on actual data]
4. **Practical Implications:** [Results to be filled based on actual data]

### Future Work

- Cross-dataset validation
- Ensemble methods
- Real-time deployment optimization
- Transfer learning from pre-trained models

## 9. Publication-Ready Figures {#9-figures}

Generate all publication-quality figures.

In [ ]:
# Generate comprehensive figure panel
if df is not None:
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    # 1. Overall accuracy distribution
    ax1 = fig.add_subplot(gs[0, :])
    ax1.hist(df['mean_accuracy'], bins=20, edgecolor='black', alpha=0.7)
    ax1.axvline(df['mean_accuracy'].mean(), color='r', linestyle='--', 
                label=f'Mean: {df["mean_accuracy"].mean():.2f}%')
    ax1.set_xlabel('Accuracy (%)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('(A) Overall Accuracy Distribution Across All Experiments')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Transform comparison
    ax2 = fig.add_subplot(gs[1, 0])
    transform_means.plot(kind='barh', ax=ax2, color=sns.color_palette('husl', len(transform_means)))
    ax2.set_xlabel('Mean Accuracy (%)')
    ax2.set_title('(B) Performance by Transform Method')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # 3. Model comparison
    ax3 = fig.add_subplot(gs[1, 1])
    model_means.plot(kind='barh', ax=ax3, color=sns.color_palette('Set2', len(model_means)))
    ax3.set_xlabel('Mean Accuracy (%)')
    ax3.set_title('(C) Performance by Model Architecture')
    ax3.grid(True, alpha=0.3, axis='x')
    
    # 4. Interaction heatmap (small)
    ax4 = fig.add_subplot(gs[2, :])
    pivot_small = df.pivot_table(values='mean_accuracy', index='transform', 
                                  columns='model', aggfunc='mean')
    sns.heatmap(pivot_small, annot=True, fmt='.1f', cmap='RdYlGn',
                center=75, ax=ax4, cbar_kws={'label': 'Accuracy (%)'})
    ax4.set_title('(D) Transform × Model Interaction')
    ax4.set_xlabel('Model')
    ax4.set_ylabel('Transform')
    
    plt.suptitle('Phase 3: Comprehensive Results Overview', fontsize=16, y=0.995)
    plt.savefig(FIGURES_DIR / 'comprehensive_overview.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Figure saved: comprehensive_overview.png")

## 10. Appendices {#10-appendices}

### A. Full Results Table

In [ ]:
if df is not None:
    # Export complete results
    results_export = df[['transform', 'model', 'subject', 'mean_accuracy', 
                         'std_accuracy', 'min_accuracy', 'max_accuracy']].copy()
    results_export = results_export.sort_values('mean_accuracy', ascending=False)
    
    display(results_export.head(20))
    
    # Save
    results_export.to_csv(RESULTS_DIR / 'complete_results_table.csv', index=False)
    print(f"\n✓ Complete results saved to: complete_results_table.csv")

### B. Statistical Test Results

In [ ]:
# Save statistical test results
if df is not None:
    stats_summary = {
        'transform_anova': {'F': float(f_stat_transform), 'p': float(p_transform)},
        'model_anova': {'F': float(f_stat_model), 'p': float(p_model)},
        'best_transform': best_transform,
        'best_model': best_model,
        'overall_mean': float(df['mean_accuracy'].mean()),
        'overall_std': float(df['mean_accuracy'].std())
    }
    
    with open(RESULTS_DIR / 'statistical_summary.json', 'w') as f:
        json.dump(stats_summary, f, indent=2)
    
    print("\n✓ Statistical summary saved to: statistical_summary.json")

---

## Analysis Complete

All figures and results have been saved to `results/phase3/`.

**Generated Files:**
- Figures: `results/phase3/figures/`
- Tables: `results/phase3/` (CSV files)
- Statistics: `results/phase3/statistical_summary.json`